# `my_env` Test Suite

Tests the **Customer Service Environment** hosted on HuggingFace Spaces.

**Server:** `https://huggingface.co/spaces/Abhimalyala/customerenv`  
**Strategy:** `.claude/docs/TESTING_STRATEGY.md`

---

## Test Hierarchy
1. **Unit Tests** — Pydantic model validation, enum coverage, reward fields
2. **Integration Tests** — Full client-server lifecycle via the HF Space WebSocket
3. **Grader Tests** — Deterministic scoring logic (grade_task)
4. **E2E Task Tests** — All three tasks (easy / medium / hard) run to completion


In [1]:
# ── 0. Install / verify dependencies ─────────────────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pydantic>=2.0", "websockets", "httpx", "openenv-core>=0.2.2"],
    check=True
)
print("✅ Dependencies ready")

✅ Dependencies ready


In [2]:
# ── 0b. Resolve project root and add to sys.path ─────────────────────────────
import os, sys, pathlib

# Works both when CWD is the project root or a subdirectory
repo_root = pathlib.Path(os.getcwd())
if not (repo_root / "my_env").exists():
    repo_root = repo_root.parent       # one level up
assert (repo_root / "my_env").exists(), f"Cannot find my_env from {repo_root}"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"📁 Repo root: {repo_root}")

📁 Repo root: c:\Users\HP\customerenviroment


In [3]:
# ── 0c. HF Space base URL ─────────────────────────────────────────────────────
# The space is running at https://Abhimalyala-customerenv.hf.space
# (HuggingFace renames spaces to <owner>-<space>.hf.space)
HF_BASE_URL  = "https://abhimalyala-customerenv.hf.space"
HF_WS_URL    = "wss://abhimalyala-customerenv.hf.space"

print(f"🌐 Remote server: {HF_BASE_URL}")

🌐 Remote server: https://abhimalyala-customerenv.hf.space


---
## 1. Unit Tests — Pydantic Models & Enums

In [4]:
# ── 1.1 Enum coverage ─────────────────────────────────────────────────────────
from my_env.models import (
    ActionType, TicketPriority, TicketCategory, TicketStatus, TaskLevel,
    CustomerServiceAction, CustomerServiceObservation, CustomerServiceState,
    CustomerServiceReward, CustomerServiceInfo, TaskGrade, TicketOutcome,
    CustomerMessage,
)

assert set(ActionType) == {
    ActionType.RESPOND, ActionType.ESCALATE, ActionType.RESOLVE,
    ActionType.REQUEST_INFO, ActionType.CATEGORIZE, ActionType.TRANSFER
}, "ActionType enum mismatch"

assert set(TicketPriority) == {
    TicketPriority.LOW, TicketPriority.MEDIUM,
    TicketPriority.HIGH, TicketPriority.URGENT
}, "TicketPriority enum mismatch"

assert set(TicketCategory) == {
    TicketCategory.BILLING, TicketCategory.TECHNICAL, TicketCategory.ACCOUNT,
    TicketCategory.PRODUCT, TicketCategory.SHIPPING, TicketCategory.GENERAL
}, "TicketCategory enum mismatch"

assert set(TicketStatus) == {
    TicketStatus.OPEN, TicketStatus.IN_PROGRESS, TicketStatus.WAITING_CUSTOMER,
    TicketStatus.ESCALATED, TicketStatus.RESOLVED, TicketStatus.CLOSED
}, "TicketStatus enum mismatch"

assert set(TaskLevel) == {TaskLevel.EASY, TaskLevel.MEDIUM, TaskLevel.HARD}

print("✅ 1.1 All enum values verified")

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 1.1 All enum values verified


In [5]:
# ── 1.2 CustomerServiceAction validation ──────────────────────────────────────
from pydantic import ValidationError

# Valid minimal action
a = CustomerServiceAction(action_type=ActionType.RESPOND, message="Hello!")
assert a.action_type == ActionType.RESPOND
assert a.category is None
assert a.priority is None
assert a.escalation_reason is None
assert a.transfer_department is None

# Valid categorize action with optional fields populated
cat_action = CustomerServiceAction(
    action_type=ActionType.CATEGORIZE,
    message="Filing under billing.",
    category=TicketCategory.BILLING,
    priority=TicketPriority.HIGH,
)
assert cat_action.category == TicketCategory.BILLING
assert cat_action.priority == TicketPriority.HIGH

# Missing required fields should raise ValidationError
try:
    CustomerServiceAction()  # type: ignore
    assert False, "Should have raised ValidationError"
except (ValidationError, TypeError):
    pass  # expected

print("✅ 1.2 CustomerServiceAction validation OK")

✅ 1.2 CustomerServiceAction validation OK


In [6]:
# ── 1.3 CustomerServiceReward default values ───────────────────────────────────
r = CustomerServiceReward()
assert r.classification == 0.0
assert r.communication == 0.0
assert r.task_progress == 0.0
assert r.resolution == 0.0
assert r.efficiency_penalty == 0.0
assert r.safety_penalty == 0.0
assert r.total == 0.0
assert 0.0 <= r.normalized_progress <= 1.0

print("✅ 1.3 CustomerServiceReward defaults OK")

✅ 1.3 CustomerServiceReward defaults OK


In [7]:
# ── 1.4 CustomerServiceState defaults ─────────────────────────────────────────
state = CustomerServiceState()
assert state.step_count == 0
assert state.cumulative_reward == 0.0
assert state.resolved is False
assert state.escalated is False
assert state.final_grade is None
assert isinstance(state.action_history, list)
assert isinstance(state.completed_tickets, list)

print("✅ 1.4 CustomerServiceState defaults OK")

✅ 1.4 CustomerServiceState defaults OK


In [8]:
# ── 1.5 TaskGrade field constraints ───────────────────────────────────────────
grade = TaskGrade(
    task_id="test",
    task_title="Test Task",
    score=0.85,
    passed=True,
)
assert 0.0 <= grade.score <= 1.0
assert grade.passed is True

# Score out-of-range should fail
try:
    TaskGrade(task_id="x", task_title="x", score=1.5, passed=False)  # type: ignore
    assert False, "Score > 1.0 should raise ValidationError"
except ValidationError:
    pass

print("✅ 1.5 TaskGrade constraints OK")

✅ 1.5 TaskGrade constraints OK


In [9]:
# ── 1.6 TicketOutcome placeholder factory ─────────────────────────────────────
placeholder = TicketOutcome.placeholder("billing_refund")
assert placeholder.scenario_id == "billing_refund"
assert placeholder.category_correct is False
assert placeholder.priority_correct is False
assert placeholder.resolved is False
assert placeholder.escalated is False
assert 0.0 <= placeholder.final_satisfaction <= 1.0

print("✅ 1.6 TicketOutcome.placeholder OK")

✅ 1.6 TicketOutcome.placeholder OK


In [10]:
# ── 1.7 CustomerServiceInfo policy_violations list ────────────────────────────
info = CustomerServiceInfo(policy_violations=["unnecessary_escalation"])
assert "unnecessary_escalation" in info.policy_violations

empty_info = CustomerServiceInfo()
assert empty_info.policy_violations == []

print("✅ 1.7 CustomerServiceInfo policy_violations OK")

✅ 1.7 CustomerServiceInfo policy_violations OK


---
## 2. Grader Tests — Deterministic Scoring Logic

In [11]:
# ── 2.1 Perfect score on easy task ────────────────────────────────────────────
from my_env.graders import grade_task
from my_env.tasks import get_task

easy_task = get_task("easy_billing_refund")
perfect_state = CustomerServiceState(
    task_id="easy_billing_refund",
    step_count=3,
    completed_tickets=[
        TicketOutcome(
            scenario_id="billing_refund",
            category_correct=True,
            priority_correct=True,
            responded=True,
            empathetic_response=True,
            issue_specific_response=True,
            resolved=True,
            unnecessary_escalation=False,
            excessive_steps=False,
            steps_taken=3,
            final_satisfaction=0.9,
        )
    ],
)

grade = grade_task(easy_task, perfect_state)
print(f"Easy task perfect score: {grade.score:.4f}")
assert 0.0 <= grade.score <= 1.0, "Score must be in [0, 1]"
assert grade.score >= 0.8, f"Perfect play should score >= 0.8, got {grade.score}"
assert grade.passed is True

print("✅ 2.1 Perfect easy task grading OK")

Easy task perfect score: 0.9100
✅ 2.1 Perfect easy task grading OK


In [12]:
# ── 2.2 Score bounds always [0.0, 1.0] with empty state ───────────────────────
for task_id in ["easy_billing_refund", "medium_support_queue", "hard_escalation_judgment"]:
    task = get_task(task_id)
    empty = CustomerServiceState(task_id=task_id, step_count=0)
    g = grade_task(task, empty)
    assert 0.0 <= g.score <= 1.0, f"{task_id}: score {g.score} out of range"
    print(f"  {task_id}: score={g.score:.4f}, passed={g.passed}")

print("✅ 2.2 Score bounds OK for all tasks")

  easy_billing_refund: score=0.3800, passed=False
  medium_support_queue: score=0.3850, passed=False
  hard_escalation_judgment: score=0.2363, passed=False
✅ 2.2 Score bounds OK for all tasks


In [13]:
# ── 2.3 Escalation accuracy is penalized when required escalation is missed ───
hard_task = get_task("hard_escalation_judgment")

# repeat_contact requires escalation — simulate it NOT being escalated
bad_state = CustomerServiceState(
    task_id="hard_escalation_judgment",
    step_count=8,
    completed_tickets=[
        TicketOutcome(
            scenario_id="repeat_contact",
            category_correct=True,
            priority_correct=True,
            responded=True,
            escalated=False,   # should have been escalated!
            resolved=True,     # incorrectly resolved
            unnecessary_escalation=False,
            final_satisfaction=0.5,
        )
    ],
)

bad_grade = grade_task(hard_task, bad_state)
print(f"Bad escalation grade: {bad_grade.score:.4f}")
assert bad_grade.score < 0.8, "Missed required escalation should not pass"

print("✅ 2.3 Escalation penalty applied correctly")

Bad escalation grade: 0.2533
✅ 2.3 Escalation penalty applied correctly


In [14]:
# ── 2.4 Grade summary is non-empty ───────────────────────────────────────────
for task_id in ["easy_billing_refund", "medium_support_queue", "hard_escalation_judgment"]:
    task = get_task(task_id)
    g = grade_task(task, CustomerServiceState(task_id=task_id))
    assert isinstance(g.summary, str) and len(g.summary) > 0
    assert g.task_id == task_id

print("✅ 2.4 Grade summaries are non-empty strings")

✅ 2.4 Grade summaries are non-empty strings


---
## 3. Task Registry Tests

In [15]:
# ── 3.1 All expected task IDs exist ──────────────────────────────────────────
from my_env.tasks import get_task, task_from_seed, TASK_REGISTRY

expected_tasks = {"easy_billing_refund", "medium_support_queue", "hard_escalation_judgment"}
assert expected_tasks == set(TASK_REGISTRY.keys()), "Task registry mismatch"

for tid in expected_tasks:
    task = get_task(tid)
    assert task.task_id == tid
    assert len(task.tickets) > 0, f"Task {tid} has no tickets"
    assert task.max_steps >= len(task.tickets), "max_steps must be >= ticket count"
    assert len(task.success_criteria) > 0
    print(f"  {tid}: {len(task.tickets)} ticket(s), max_steps={task.max_steps}")

print("✅ 3.1 All task IDs and specs valid")

  hard_escalation_judgment: 5 ticket(s), max_steps=16
  easy_billing_refund: 1 ticket(s), max_steps=4
  medium_support_queue: 3 ticket(s), max_steps=10
✅ 3.1 All task IDs and specs valid


In [16]:
# ── 3.2 task_from_seed is deterministic ───────────────────────────────────────
t1 = task_from_seed(0)
t2 = task_from_seed(0)
assert t1.task_id == t2.task_id, "task_from_seed must be deterministic"

# Cycling: seed % len maps into valid tasks
for seed in range(10):
    t = task_from_seed(seed)
    assert t.task_id in TASK_REGISTRY

print("✅ 3.2 task_from_seed is deterministic")

✅ 3.2 task_from_seed is deterministic


In [17]:
# ── 3.3 TicketSpec fields are sensible ────────────────────────────────────────
from my_env.models import TicketCategory, TicketPriority

for task in TASK_REGISTRY.values():
    seen_ids = set()
    for ticket in task.tickets:
        assert ticket.scenario_id not in seen_ids, "Duplicate scenario_id in task"
        seen_ids.add(ticket.scenario_id)
        assert isinstance(ticket.true_category, TicketCategory)
        assert isinstance(ticket.true_priority, TicketPriority)
        assert len(ticket.customer_name) > 0
        assert len(ticket.opening_message) > 0
        assert ticket.sentiment in {"positive", "neutral", "negative", "angry"}

print("✅ 3.3 All TicketSpec fields are valid")

✅ 3.3 All TicketSpec fields are valid


---
## 4. Integration Tests — Local Environment (Direct)

These tests use `CustomerServiceEnvironment` directly (no network), validating the full reset → step → done lifecycle.

In [18]:
# ── 4.1 Easy task: perfect play scores 1.0 ────────────────────────────────────
from my_env.server.my_env_environment import CustomerServiceEnvironment

env = CustomerServiceEnvironment(task_id="easy_billing_refund")
obs = env.reset(episode_id="easy_billing_refund")

assert obs is not None, "reset() must return an observation"
assert obs.task_id == "easy_billing_refund"
assert obs.done is False

# Step 1: Categorize
env.step(CustomerServiceAction(
    action_type=ActionType.CATEGORIZE,
    message="Classifying the duplicate charge.",
    category=TicketCategory.BILLING,
    priority=TicketPriority.HIGH,
))

# Step 2: Respond empathetically
env.step(CustomerServiceAction(
    action_type=ActionType.RESPOND,
    message="I am sorry about the duplicate charge. I will refund the extra payment.",
))

# Step 3: Resolve
final_obs = env.step(CustomerServiceAction(
    action_type=ActionType.RESOLVE,
    message="The refund has been submitted and your ticket is resolved.",
))

assert final_obs.done is True, "Episode should be done after resolve"
assert env.state.final_grade is not None
assert env.state.final_grade.score == 1.0, f"Expected score >= 0.8, got {env.state.final_grade.score}"

print(f"✅ 4.1 Easy task score: {env.state.final_grade.score:.4f}")

AssertionError: Expected score >= 0.8, got 0.899

In [ ]:
# ── 4.2 Medium task: partial progress rewards request_info ────────────────────
env = CustomerServiceEnvironment(task_id="medium_support_queue")
env.reset(episode_id="medium_support_queue")

env.step(CustomerServiceAction(
    action_type=ActionType.CATEGORIZE,
    message="Routing this login issue.",
    category=TicketCategory.ACCOUNT,
    priority=TicketPriority.HIGH,
))

obs = env.step(CustomerServiceAction(
    action_type=ActionType.REQUEST_INFO,
    message="Please confirm the email address tied to the locked account.",
))

assert obs.reward > 0.0, "Correct request_info should yield positive reward"
assert obs.reward_details.task_progress > 0.0
assert obs.status.value in {"waiting_customer", "in_progress"}

print(f"✅ 4.2 Medium task partial progress reward: {obs.reward:.4f}")

✅ 4.2 Medium task partial progress reward: 0.1500


In [ ]:
# ── 4.3 Hard task: unnecessary escalation is penalized ────────────────────────
env = CustomerServiceEnvironment(task_id="hard_escalation_judgment")
env.reset(episode_id="hard_escalation_judgment")

# Ticket 1: repeat_contact — requires escalation (correct)
env.step(CustomerServiceAction(
    action_type=ActionType.CATEGORIZE,
    message="Routing repeat-contact complaint.",
    category=TicketCategory.GENERAL,
    priority=TicketPriority.URGENT,
))
env.step(CustomerServiceAction(
    action_type=ActionType.RESPOND,
    message="I am sorry this has been so frustrating. I am escalating this with priority.",
))
env.step(CustomerServiceAction(
    action_type=ActionType.ESCALATE,
    message="Escalating to Tier-2.",
    escalation_reason="Repeat contact with unresolved issue",
))

# Ticket 2: late_shipment — should NOT be escalated, it's resolvable
env.step(CustomerServiceAction(
    action_type=ActionType.CATEGORIZE,
    message="Routing the late shipment issue.",
    category=TicketCategory.SHIPPING,
    priority=TicketPriority.HIGH,
))
obs = env.step(CustomerServiceAction(
    action_type=ActionType.ESCALATE,   # unnecessary escalation!
    message="Escalating the shipping issue.",
    escalation_reason="Delayed order",
))

assert obs.reward < 0.0, f"Unnecessary escalation must yield negative reward, got {obs.reward}"
assert "unnecessary_escalation" in obs.info.policy_violations

print(f"✅ 4.3 Hard task unnecessary escalation penalty: {obs.reward:.4f}")

✅ 4.3 Hard task unnecessary escalation penalty: -0.2500


: 

In [ ]:
# ── 4.4 Reset idempotency — episode restarts cleanly ─────────────────────────
env = CustomerServiceEnvironment(task_id="easy_billing_refund")

obs1 = env.reset(episode_id="run1")
env.step(CustomerServiceAction(action_type=ActionType.RESPOND, message="Hi!"))
assert env.state.step_count > 0

obs2 = env.reset(episode_id="run2")  # reset again
assert env.state.step_count == 0, "Step count must reset to 0"
assert env.state.cumulative_reward == 0.0, "Cumulative reward must reset"
assert obs2.done is False

print("✅ 4.4 Reset idempotency OK")

In [ ]:
# ── 4.5 Step-limit enforcement ────────────────────────────────────────────────
# The easy task has max_steps=4. After 4 steps without resolve the env should end.
env = CustomerServiceEnvironment(task_id="easy_billing_refund")
env.reset(episode_id="step_limit_test")

done = False
for i in range(15):   # far more than the task allows
    obs = env.step(CustomerServiceAction(
        action_type=ActionType.RESPOND,
        message=f"Still working on it, step {i}.",
    ))
    if obs.done:
        done = True
        print(f"  Episode ended at step {env.state.step_count}")
        break

assert done, "Environment must enforce a step limit and set done=True"
print("✅ 4.5 Step-limit enforced")

In [ ]:
# ── 4.6 Observation fields are well-typed after reset ─────────────────────────
env = CustomerServiceEnvironment(task_id="medium_support_queue")
obs = env.reset(episode_id="field_type_check")

assert isinstance(obs.ticket_id, str) and obs.ticket_id
assert isinstance(obs.customer_name, str) and obs.customer_name
assert isinstance(obs.customer_message, str) and obs.customer_message
assert isinstance(obs.available_actions, list) and len(obs.available_actions) > 0
assert all(isinstance(a, ActionType) for a in obs.available_actions)
assert isinstance(obs.conversation_history, list)
assert isinstance(obs.status, TicketStatus)
assert isinstance(obs.priority, TicketPriority)
assert obs.done is False

print("✅ 4.6 Observation field types OK")

---
## 5. E2E Task Tests — All Three Tasks Run to Completion

In [ ]:
# ── 5.1 Easy task full run ────────────────────────────────────────────────────
env = CustomerServiceEnvironment(task_id="easy_billing_refund")
env.reset(episode_id="e2e_easy")

actions = [
    CustomerServiceAction(
        action_type=ActionType.CATEGORIZE,
        message="This is a billing duplicate charge ticket.",
        category=TicketCategory.BILLING,
        priority=TicketPriority.HIGH,
    ),
    CustomerServiceAction(
        action_type=ActionType.RESPOND,
        message="I'm sorry for the duplicate charge. We will process a full refund immediately.",
    ),
    CustomerServiceAction(
        action_type=ActionType.RESOLVE,
        message="The refund has been processed. Your ticket is now resolved.",
    ),
]

final = None
for action in actions:
    final = env.step(action)

assert final.done is True
grade = env.state.final_grade
assert grade is not None
print(f"📊 Easy task — score: {grade.score:.4f}, passed: {grade.passed}")
print(f"   Per-ticket: {grade.per_ticket_scores}")
print(f"   Summary: {grade.summary}")
assert 0.0 <= grade.score <= 1.0

print("✅ 5.1 Easy task E2E complete")

In [ ]:
# ── 5.2 Medium task full run ───────────────────────────────────────────────────
env = CustomerServiceEnvironment(task_id="medium_support_queue")
env.reset(episode_id="e2e_medium")

# Ticket 1: account_lockout — needs REQUEST_INFO
ticket1_actions = [
    CustomerServiceAction(
        action_type=ActionType.CATEGORIZE,
        message="Login / account access issue.",
        category=TicketCategory.ACCOUNT,
        priority=TicketPriority.HIGH,
    ),
    CustomerServiceAction(
        action_type=ActionType.REQUEST_INFO,
        message="Could you confirm the email address on the account so I can unlock it?",
    ),
    CustomerServiceAction(
        action_type=ActionType.RESPOND,
        message="Thank you. I have unlocked your account — please try logging in with your email.",
    ),
    CustomerServiceAction(
        action_type=ActionType.RESOLVE,
        message="Account lockout resolved.",
    ),
]
# Ticket 2: csv_export — technical, low priority
ticket2_actions = [
    CustomerServiceAction(
        action_type=ActionType.CATEGORIZE,
        message="Technical product usage question.",
        category=TicketCategory.TECHNICAL,
        priority=TicketPriority.LOW,
    ),
    CustomerServiceAction(
        action_type=ActionType.RESPOND,
        message="You can export to CSV via Settings > Account > Export. Let me know if you need help!",
    ),
    CustomerServiceAction(
        action_type=ActionType.RESOLVE,
        message="CSV export question answered.",
    ),
]
# Ticket 3: checkout_error — billing, medium priority
ticket3_actions = [
    CustomerServiceAction(
        action_type=ActionType.CATEGORIZE,
        message="Billing / checkout error.",
        category=TicketCategory.BILLING,
        priority=TicketPriority.MEDIUM,
    ),
    CustomerServiceAction(
        action_type=ActionType.RESPOND,
        message="I see the checkout issue. We can process a manual upgrade — here are the steps.",
    ),
    CustomerServiceAction(
        action_type=ActionType.RESOLVE,
        message="Upgrade processed manually. Checkout issue resolved.",
    ),
]

final_obs = None
for action in ticket1_actions + ticket2_actions + ticket3_actions:
    final_obs = env.step(action)
    if final_obs.done:
        break

assert final_obs is not None and final_obs.done is True
grade = env.state.final_grade
assert grade is not None
print(f"📊 Medium task — score: {grade.score:.4f}, passed: {grade.passed}")
print(f"   Per-ticket: {grade.per_ticket_scores}")
print(f"   Summary: {grade.summary}")
assert 0.0 <= grade.score <= 1.0

print("✅ 5.2 Medium task E2E complete")

In [ ]:
# ── 5.3 Hard task full run ────────────────────────────────────────────────────
env = CustomerServiceEnvironment(task_id="hard_escalation_judgment")
env.reset(episode_id="e2e_hard")

hard_actions = [
    # Ticket 1: repeat_contact — must escalate
    CustomerServiceAction(
        action_type=ActionType.CATEGORIZE,
        message="Repeat contact complaint.",
        category=TicketCategory.GENERAL,
        priority=TicketPriority.URGENT,
    ),
    CustomerServiceAction(
        action_type=ActionType.RESPOND,
        message="I am so sorry for the repeated issues — I'm escalating this to our Tier-2 team as a priority.",
    ),
    CustomerServiceAction(
        action_type=ActionType.ESCALATE,
        message="Escalating to Tier-2.",
        escalation_reason="Repeat contact, unresolved issue requires senior attention",
    ),
    # Ticket 2: late_shipment — respond & resolve (no escalation)
    CustomerServiceAction(
        action_type=ActionType.CATEGORIZE,
        message="Late shipment complaint.",
        category=TicketCategory.SHIPPING,
        priority=TicketPriority.HIGH,
    ),
    CustomerServiceAction(
        action_type=ActionType.RESPOND,
        message="I apologize for the delay on order #4521. We will arrange a replacement or refund for you.",
    ),
    CustomerServiceAction(action_type=ActionType.RESOLVE, message="Replacement/refund initiated."),
    # Ticket 3: dashboard_bug — transfer to product_feedback
    CustomerServiceAction(
        action_type=ActionType.CATEGORIZE,
        message="Product feedback.",
        category=TicketCategory.PRODUCT,
        priority=TicketPriority.LOW,
    ),
    CustomerServiceAction(
        action_type=ActionType.RESPOND,
        message="Thanks for the feedback! I can see the charts bug you mentioned and will log it.",
    ),
    CustomerServiceAction(
        action_type=ActionType.TRANSFER,
        message="Transferring to product feedback team.",
        transfer_department="product_feedback",
    ),
    # Ticket 4: enterprise_features — transfer to sales
    CustomerServiceAction(
        action_type=ActionType.CATEGORIZE,
        message="Enterprise inquiry.",
        category=TicketCategory.PRODUCT,
        priority=TicketPriority.LOW,
    ),
    CustomerServiceAction(
        action_type=ActionType.RESPOND,
        message="Our enterprise plan includes advanced features. Let me transfer you to our sales team.",
    ),
    CustomerServiceAction(
        action_type=ActionType.TRANSFER,
        message="Transferring to sales.",
        transfer_department="sales",
    ),
    # Ticket 5: billing_refund_repeatable — respond & resolve
    CustomerServiceAction(
        action_type=ActionType.CATEGORIZE,
        message="Billing duplicate charge.",
        category=TicketCategory.BILLING,
        priority=TicketPriority.HIGH,
    ),
    CustomerServiceAction(
        action_type=ActionType.RESPOND,
        message="I'm sorry for the duplicate charge. A refund has been initiated for the extra payment.",
    ),
    CustomerServiceAction(action_type=ActionType.RESOLVE, message="Refund processed. Ticket resolved."),
]

final_obs = None
for action in hard_actions:
    final_obs = env.step(action)
    if final_obs.done:
        break

assert final_obs is not None and final_obs.done is True
grade = env.state.final_grade
assert grade is not None
print(f"📊 Hard task — score: {grade.score:.4f}, passed: {grade.passed}")
print(f"   Per-ticket: {grade.per_ticket_scores}")
print(f"   Summary: {grade.summary}")
assert 0.0 <= grade.score <= 1.0

print("✅ 5.3 Hard task E2E complete")

---
## 6. Remote HF Space Tests — HTTP Endpoints

Validates that the HuggingFace Space at `https://abhimalyala-customerenv.hf.space` is reachable and responds correctly.

In [ ]:
# ── 6.1 Health / schema endpoint reachable ────────────────────────────────────
import httpx

try:
    resp = httpx.get(f"{HF_BASE_URL}/schema", timeout=30)
    print(f"GET /schema → {resp.status_code}")
    if resp.status_code == 200:
        schema = resp.json()
        print(f"   Keys: {list(schema.keys())}")
        print("✅ 6.1 /schema reachable")
    else:
        print(f"⚠️  /schema returned {resp.status_code} — space may be sleeping")
except httpx.ConnectError as e:
    print(f"⚠️  Cannot reach HF Space: {e}")
    print("   (Space may be sleeping or URL is wrong — local tests still pass)")

In [ ]:
# ── 6.2 POST /reset on HF Space ───────────────────────────────────────────────
import json

try:
    resp = httpx.post(
        f"{HF_BASE_URL}/reset",
        json={"task_id": "easy_billing_refund", "episode_id": "hf_test_easy"},
        timeout=30,
    )
    print(f"POST /reset → {resp.status_code}")
    if resp.status_code == 200:
        obs_payload = resp.json()
        print(f"   task_id: {obs_payload.get('task_id', obs_payload.get('observation', {}).get('task_id', 'N/A'))}")
        print("✅ 6.2 /reset OK")
    else:
        print(f"⚠️  /reset returned {resp.status_code}: {resp.text[:200]}")
except Exception as e:
    print(f"⚠️  /reset not reachable: {e}")

In [ ]:
# ── 6.3 POST /step on HF Space ────────────────────────────────────────────────
try:
    # First reset
    reset_resp = httpx.post(
        f"{HF_BASE_URL}/reset",
        json={"task_id": "easy_billing_refund", "episode_id": "hf_step_test"},
        timeout=30,
    )
    if reset_resp.status_code != 200:
        print(f"⚠️  /reset failed ({reset_resp.status_code}), skipping step test")
    else:
        step_resp = httpx.post(
            f"{HF_BASE_URL}/step",
            json={
                "action_type": "categorize",
                "message": "Classifying billing ticket.",
                "category": "billing",
                "priority": "high",
            },
            timeout=30,
        )
        print(f"POST /step → {step_resp.status_code}")
        if step_resp.status_code == 200:
            result = step_resp.json()
            reward = result.get("reward", "N/A")
            done   = result.get("done", "N/A")
            print(f"   reward={reward}, done={done}")
            print("✅ 6.3 /step OK")
        else:
            print(f"⚠️  /step returned {step_resp.status_code}: {step_resp.text[:200]}")
except Exception as e:
    print(f"⚠️  /step not reachable: {e}")

In [ ]:
# ── 6.4 GET /state on HF Space ────────────────────────────────────────────────
try:
    resp = httpx.get(f"{HF_BASE_URL}/state", timeout=30)
    print(f"GET /state → {resp.status_code}")
    if resp.status_code == 200:
        state_data = resp.json()
        print(f"   step_count={state_data.get('step_count', 'N/A')}")
        print("✅ 6.4 /state OK")
    else:
        print(f"⚠️  /state returned {resp.status_code}")
except Exception as e:
    print(f"⚠️  /state not reachable: {e}")

---
## 7. Test Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                  my_env Test Suite Results                      ║
╠══════════════════════════════════════════════════════════════════╣
║  Section   │ Tests  │ Description                               ║
║────────────┼────────┼───────────────────────────────────────────║
║  1. Unit   │  7     │ Pydantic models, enums, field constraints ║
║  2. Grader │  4     │ Deterministic grade_task scoring logic    ║
║  3. Tasks  │  3     │ Task registry, TicketSpec validation      ║
║  4. Integ. │  6     │ Local env lifecycle (reset/step/done)     ║
║  5. E2E    │  3     │ All three tasks run to completion         ║
║  6. Remote │  4     │ HF Space HTTP endpoints (graceful skip)   ║
╚══════════════════════════════════════════════════════════════════╝
"""
)